## Human Activity Recognition using Random Forest
## IT22341822 - K M H G S A KONARA

**Dataset:** Human Activity Recognition Using Smartphones  
**Source:** https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones  
**Algorithm:** Random Forest Classifier  
**Language:** Python 3  
**Environment:** Jupyter Notebook

## Dataset Description

The Human Activity Recognition (HAR) Using Smartphones dataset was built from experiments carried out with a group of 30 volunteers aged 19–48 years. Each person performed six activities:
| 1 | WALKING |
| 2 | WALKING_UPSTAIRS |
| 3 | WALKING_DOWNSTAIRS |
| 4 | SITTING |
| 5 | STANDING |
| 6 | LAYING |

Wearing a smartphone on the waist, the embedded accelerometer and gyroscope captured:
- 3-axial linear acceleration
- 3-axial angular velocity

Sensor signals were pre-processed and a feature vector of 561 attributes was extracted.
Dataset Size:
- Training set: 7,352 instances
- Test set: 2,947 instances
- Total features: 561
- Classes: 6

## Link: https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score
)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.inspection import permutation_importance

# Display settings
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

print("All libraries imported successfully!")

All libraries imported successfully!


### Load the Dataset

In [2]:
BASE_PATH = r"C:\Users\Sadali\Desktop\assignment_ML\human+activity+recognition+using+smartphones\UCI HAR Dataset\UCI HAR Dataset"

import os

def load_har_dataset(base_path):
    """
    Load the UCI HAR dataset from the given base path.
    Returns X_train, X_test, y_train, y_test, feature_names, activity_labels.
    """
    # feature names
    features_path = os.path.join(base_path, 'features.txt')
    features = pd.read_csv(features_path, sep='\s+', header=None, names=['idx', 'feature'])
    feature_names = features['feature'].tolist()

    # activity labels
    labels_path = os.path.join(base_path, 'activity_labels.txt')
    activity_labels = pd.read_csv(labels_path, sep='\s+', header=None, names=['id', 'activity'])
    activity_map = dict(zip(activity_labels['id'], activity_labels['activity']))

    # training data
    X_train = pd.read_csv(os.path.join(base_path, 'train', 'X_train.txt'), sep='\s+', header=None)
    y_train = pd.read_csv(os.path.join(base_path, 'train', 'y_train.txt'), sep='\s+', header=None, names=['activity'])

    # test data
    X_test = pd.read_csv(os.path.join(base_path, 'test', 'X_test.txt'), sep='\s+', header=None)
    y_test = pd.read_csv(os.path.join(base_path, 'test', 'y_test.txt'), sep='\s+', header=None, names=['activity'])

    # feature names
    X_train.columns = feature_names
    X_test.columns  = feature_names

    # Map numeric labels to activity names
    y_train['activity'] = y_train['activity'].map(activity_map)
    y_test['activity']  = y_test['activity'].map(activity_map)

    return X_train, X_test, y_train['activity'], y_test['activity'], feature_names, activity_map

# Load the data
X_train, X_test, y_train, y_test, feature_names, activity_map = load_har_dataset(BASE_PATH)

print(f"Training set shape : {X_train.shape}")
print(f"Test set shape     : {X_test.shape}")
print(f"Number of features : {X_train.shape[1]}")
print(f"Activity classes   : {list(activity_map.values())}")

Training set shape : (7352, 561)
Test set shape     : (2947, 561)
Number of features : 561
Activity classes   : ['WALKING', 'WALKING_UPSTAIRS', 'WALKING_DOWNSTAIRS', 'SITTING', 'STANDING', 'LAYING']


## Exploratory Data Analysis (EDA)

In [ ]:
# ----  Class Distribution ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

y_train.value_counts().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Training Set - Class Distribution', fontsize=13)
axes[0].set_xlabel('Activity')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

y_test.value_counts().plot(kind='bar', ax=axes[1], color='coral', edgecolor='black')
axes[1].set_title('Test Set - Class Distribution', fontsize=13)
axes[1].set_xlabel('Activity')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

print("\nTraining Class Distribution:")
print(y_train.value_counts())

In [ ]:
# ---- Basic Statistics ----
print("Training Data Statistics (first 5 features):")
X_train.iloc[:, :5].describe().round(4)

In [ ]:
# ---- 4.3 Missing Values Check ----
missing_train = X_train.isnull().sum().sum()
missing_test  = X_test.isnull().sum().sum()

print(f"Missing values in training set : {missing_train}")
print(f"Missing values in test set     : {missing_test}")

In [ ]:
# ---- 4.4 Feature Value Range ----
print(f"Feature value range (train): [{X_train.values.min():.4f}, {X_train.values.max():.4f}]")
print("(Values are already normalized between -1 and 1 by the dataset authors)")

## Data Preprocessing

The UCI HAR dataset has been pre-processed by the authors:
- Noise filters applied to sensor signals
- Sliding window segmentation (2.56s windows, 50% overlap)
- Features normalized to [-1, 1]

1. **Duplicate feature name handling** – some feature names repeat; we make them unique
2. **Label encoding** – convert string activity labels to integers for sklearn
3. **Duplicate row check**
4. **Optional PCA** – visualise data in 2D

In [ ]:
# ---- 1 Handle Duplicate Feature Names ----
# Some features share the same name; make them unique
seen = {}
unique_names = []
for name in feature_names:
    if name in seen:
        seen[name] += 1
        unique_names.append(f"{name}_{seen[name]}")
    else:
        seen[name] = 0
        unique_names.append(name)

X_train.columns = unique_names
X_test.columns  = unique_names

duplicates = len(feature_names) - len(set(feature_names))
print(f"Duplicate feature names resolved: {duplicates}")

In [ ]:
# ---- 2 Label Encoding ----
le = LabelEncoder()
le.fit(y_train)
y_train_enc = le.transform(y_train)
y_test_enc  = le.transform(y_test)

print("Label Encoding Mapping:")
for i, cls in enumerate(le.classes_):
    print(f"  {i} -> {cls}")

In [ ]:
# ---- 3 Check Duplicate Rows ----
dup_train = X_train.duplicated().sum()
dup_test  = X_test.duplicated().sum()
print(f"Duplicate rows in training set : {dup_train}")
print(f"Duplicate rows in test set     : {dup_test}")

In [ ]:
# ---- 4 PCA Visualization (2D) ----
pca = PCA(n_components=2, random_state=42)
X_train_pca = pca.fit_transform(X_train)

plt.figure(figsize=(10, 7))
colors = plt.cm.tab10(np.linspace(0, 1, len(le.classes_)))

for i, activity in enumerate(le.classes_):
    mask = y_train == activity
    plt.scatter(X_train_pca[mask, 0], X_train_pca[mask, 1],
                label=activity, alpha=0.4, s=15, color=colors[i])

plt.title('PCA - 2D Visualisation of HAR Dataset', fontsize=14)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

print(f"Total variance explained by 2 PCs: {sum(pca.explained_variance_ratio_)*100:.2f}%")